# Ablation Study — PCL Classifier

This notebook runs a **rigorous component-level ablation** of the final ensemble pipeline.
Each run trains a DistilBERT model with exactly one component removed, then evaluates on
the official dev set (2,094 rows).

**Backbone:** `distilbert-base-uncased` (faster than RoBERTa; same qualitative conclusions)  
**Seed:** 42 (fixed for all variants to control for randomness)  
**Evaluation:** F1 of the positive (PCL) class on the official dev set

---

## Components under test

| # | Variant | What is removed / changed |
|---|---|---|
| 0 | **Full model** | All components enabled (baseline for this ablation) |
| 1 | **w/o Metadata** | Remove `<e>keyword</e> <e>country</e>` prefix; pass raw text only |
| 2 | **w/o WRS** | Replace WeightedRandomSampler with standard uniform DataLoader |
| 3 | **w/o LLRD** | Replace grouped layer-wise LR decay with a single uniform LR |
| 4 | **w/o Warmup** | Remove linear warmup; use cosine decay from step 0 |
| 5 | **Fixed threshold** | Skip threshold tuning; always classify at t = 0.50 |

Results are saved to `5_evaluation/ablation_results/` and plotted at the end.

In [ ]:
import os, sys, html, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import get_cosine_schedule_with_warmup
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from tqdm import tqdm

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

# ── Paths ─────────────────────────────────────────────────────────────────────
# Notebook lives in  <repo>/5_evaluation/  so REPO_ROOT is one level up.
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
REPO_ROOT    = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
DATA_DIR     = os.path.join(REPO_ROOT, 'data')
MODEL_DIR    = os.path.join(REPO_ROOT, '4_model')
OUT_DIR      = os.path.join(NOTEBOOK_DIR, 'ablation_results')   # saved next to the notebook
os.makedirs(OUT_DIR, exist_ok=True)
sys.path.insert(0, MODEL_DIR)
from dont_patronize_me import DontPatronizeMe

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else
                      ('mps'  if torch.backends.mps.is_available() else 'cpu'))
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

# ── Fixed seed (used for all variants) ───────────────────────────────────────
ABLATION_SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

print(f'Ablation seed: {ABLATION_SEED}')
print(f'Output dir   : {OUT_DIR}')

## 1. Data Loading

One shared load — all variants use the same train/val/test splits.

In [ ]:
dpm = DontPatronizeMe(DATA_DIR, os.path.join(DATA_DIR, 'task4_test.tsv'))
dpm.load_task1()
full_df = dpm.train_task1_df.copy()
full_df['par_id']     = full_df['par_id'].astype(str)
full_df['orig_label'] = full_df['orig_label'].astype(int)

train_ids = pd.read_csv(os.path.join(DATA_DIR, 'train_semeval_parids-labels.csv'))
dev_ids   = pd.read_csv(os.path.join(DATA_DIR, 'dev_semeval_parids-labels.csv'))
train_ids['par_id'] = train_ids['par_id'].astype(str)
dev_ids['par_id']   = dev_ids['par_id'].astype(str)

train_pool = full_df[full_df['par_id'].isin(train_ids['par_id'])].reset_index(drop=True)
test_df    = full_df[full_df['par_id'].isin(dev_ids['par_id'])].reset_index(drop=True)
test_df['text'] = test_df['text'].fillna('')

# Fixed stratified 90/10 split (same seed across ALL variants)
train_df, val_df = train_test_split(
    train_pool, test_size=0.10, stratify=train_pool['label'], random_state=ABLATION_SEED
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f'Train : {len(train_df):,}  |  PCL: {train_df.label.sum()} ({train_df.label.mean():.1%})')
print(f'Val   : {len(val_df):,}   |  PCL: {val_df.label.sum()} ({val_df.label.mean():.1%})')
print(f'Test  : {len(test_df):,}  |  PCL: {test_df.label.sum()} ({test_df.label.mean():.1%})')

## 2. Input Builders

Two input builders — **with metadata** (full model) and **without metadata** (ablation variant 1).

In [ ]:
def build_with_metadata(row):
    """Full input: <e>keyword</e> <e>country</e> {text}"""
    return f"<e>{row['keyword']}</e> <e>{row['country']}</e> {html.unescape(str(row['text']))}"

def build_no_metadata(row):
    """Ablation: raw text only, no keyword/country context."""
    return html.unescape(str(row['text']))

## 3. Dataset and DataLoader Helpers

In [ ]:
class PCLDataset(Dataset):
    def __init__(self, texts, tokenizer, labels=None, max_len=256):
        self.enc    = tokenizer(list(texts), padding='max_length', truncation=True,
                                max_length=max_len, return_tensors='pt')
        self.labels = labels

    def __len__(self):
        return self.enc['input_ids'].shape[0]

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[i], dtype=torch.long)
        return item


def make_train_loader(dataset, labels, batch_size, use_wrs=True):
    """Build a DataLoader with optional WeightedRandomSampler."""
    if use_wrs:
        class_counts = np.bincount(labels)
        weights = 1.0 / np.sqrt(class_counts[labels].astype(float))
        sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
        return DataLoader(dataset, batch_size=batch_size, sampler=sampler)
    else:
        return DataLoader(dataset, batch_size=batch_size, shuffle=True)

## 4. Optimiser Builders

Two strategies:
- **Grouped LLRD** — lower layers get lower LR; upper layers + head get higher LR.
- **Uniform LR** — all parameters share the same learning rate (ablation variant 3).

In [ ]:
def make_optimizer_llrd(model, base_lr=2e-5, head_lr=5e-5, decay=0.9):
    """
    Grouped Layer-wise Learning Rate Decay.
    DistilBERT has 6 transformer layers (layer 0..5).
    Layer i gets lr = base_lr * decay^(5 - i).
    The classifier head gets head_lr.
    """
    param_groups = []

    # Embedding layer — lowest LR
    param_groups.append({
        'params': [p for n, p in model.named_parameters() if 'embeddings' in n],
        'lr': base_lr * (decay ** 6)
    })

    # Transformer layers 0..5
    for layer_idx in range(6):
        layer_lr = base_lr * (decay ** (5 - layer_idx))
        param_groups.append({
            'params': [
                p for n, p in model.named_parameters()
                if f'layer.{layer_idx}.' in n
            ],
            'lr': layer_lr
        })

    # Classifier head
    param_groups.append({
        'params': [
            p for n, p in model.named_parameters()
            if any(k in n for k in ('classifier', 'pre_classifier', 'pooler'))
        ],
        'lr': head_lr
    })

    return AdamW(param_groups, weight_decay=0.01)


def make_optimizer_uniform(model, lr=2e-5):
    """Single uniform learning rate across all parameters — ablation for LLRD."""
    return AdamW(model.parameters(), lr=lr, weight_decay=0.01)

## 5. Scheduler Builders

- **With warmup** (full model): linear warmup for 6% of total steps, then cosine decay.
- **Without warmup** (ablation variant 4): cosine decay starting from step 0.

In [ ]:
def make_scheduler(optimizer, total_steps, use_warmup=True, warmup_frac=0.06):
    warmup_steps = int(total_steps * warmup_frac) if use_warmup else 0
    return get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

## 6. Threshold Tuning Helper

In [ ]:
def tune_threshold(probs, labels):
    """Sweep thresholds 0.05..0.95; return the one maximising F1."""
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.05, 0.95, 0.01):
        preds = (probs >= t).astype(int)
        f1 = f1_score(labels, preds, pos_label=1, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return round(float(best_t), 2), round(float(best_f1), 4)


def get_probs(model, loader):
    """Run inference; return numpy array of P(PCL) for each example."""
    model.eval()
    all_probs = []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items() if k != 'labels'}
            logits = model(**batch).logits
            all_probs.append(torch.softmax(logits, dim=-1)[:, 1].cpu().numpy())
    return np.concatenate(all_probs)

## 7. Core Training Loop

In [ ]:
def train_and_evaluate(
    variant_name,
    *,
    use_metadata   = True,
    use_wrs        = True,
    use_llrd       = True,
    use_warmup     = True,
    use_thresh_tune= True,
    model_name     = 'distilbert-base-uncased',
    dropout        = 0.3,
    batch_size     = 8,
    grad_accum     = 2,
    max_epochs     = 10,
    patience       = 3,
    max_len        = 256,
    seed           = ABLATION_SEED,
):
    """
    Train one ablation variant and evaluate on the dev set.

    Returns a dict with keys:
        variant, val_f1, val_threshold, test_f1, test_precision,
        test_recall, best_epoch
    """
    print(f'\n{"="*60}')
    print(f'  Variant : {variant_name}')
    print(f'  metadata={use_metadata}  wrs={use_wrs}  llrd={use_llrd}  '
          f'warmup={use_warmup}  thresh_tune={use_thresh_tune}')
    print(f'{"="*60}')

    set_seed(seed)

    # ── Build input texts ────────────────────────────────────────────────────
    builder = build_with_metadata if use_metadata else build_no_metadata
    tr_texts  = train_df.apply(builder, axis=1).tolist()
    vl_texts  = val_df.apply(builder, axis=1).tolist()
    te_texts  = test_df.apply(builder, axis=1).tolist()
    tr_labels = train_df['label'].tolist()
    vl_labels = val_df['label'].tolist()
    te_labels = test_df['label'].tolist()

    # ── Tokenizer and datasets ───────────────────────────────────────────────
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tr_ds = PCLDataset(tr_texts, tokenizer, labels=tr_labels, max_len=max_len)
    vl_ds = PCLDataset(vl_texts, tokenizer, labels=vl_labels, max_len=max_len)
    te_ds = PCLDataset(te_texts, tokenizer, labels=te_labels, max_len=max_len)

    train_loader = make_train_loader(tr_ds, tr_labels, batch_size, use_wrs=use_wrs)
    val_loader   = DataLoader(vl_ds, batch_size=batch_size * 2, shuffle=False)
    test_loader  = DataLoader(te_ds, batch_size=batch_size * 2, shuffle=False)

    # ── Model ────────────────────────────────────────────────────────────────
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        hidden_dropout_prob=dropout,
        attention_probs_dropout_prob=dropout,
    ).to(DEVICE)

    # ── Optimiser ────────────────────────────────────────────────────────────
    if use_llrd:
        optimizer = make_optimizer_llrd(model)
    else:
        optimizer = make_optimizer_uniform(model)

    # ── Scheduler ────────────────────────────────────────────────────────────
    steps_per_epoch = len(train_loader) // grad_accum
    total_steps     = max_epochs * steps_per_epoch
    scheduler = make_scheduler(optimizer, total_steps, use_warmup=use_warmup)

    # ── Training ─────────────────────────────────────────────────────────────
    best_val_f1     = 0.0
    best_val_probs  = None
    no_improve      = 0
    best_epoch      = 0
    best_state      = None

    for epoch in range(1, max_epochs + 1):
        model.train()
        optimizer.zero_grad()
        total_loss, step = 0.0, 0

        for i, batch in enumerate(tqdm(train_loader, desc=f'  Epoch {epoch}', leave=False)):
            batch   = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(**batch)
            loss    = outputs.loss / grad_accum
            loss.backward()
            total_loss += loss.item() * grad_accum

            if (i + 1) % grad_accum == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                step += 1

        # ── Validation ───────────────────────────────────────────────────────
        val_probs  = get_probs(model, val_loader)
        val_labels = np.array(vl_labels)
        # Always tune threshold on val for fair evaluation
        _, epoch_val_f1 = tune_threshold(val_probs, val_labels)

        avg_loss = total_loss / len(train_loader)
        print(f'  Epoch {epoch:2d} | loss={avg_loss:.4f} | val F1*={epoch_val_f1:.4f}')

        if epoch_val_f1 > best_val_f1:
            best_val_f1    = epoch_val_f1
            best_val_probs = val_probs.copy()
            best_state     = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_epoch     = epoch
            no_improve     = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  Early stop at epoch {epoch} (no improvement for {patience} epochs)')
                break

    # ── Threshold selection ──────────────────────────────────────────────────
    val_labels_arr = np.array(vl_labels)
    if use_thresh_tune:
        val_threshold, val_f1 = tune_threshold(best_val_probs, val_labels_arr)
    else:
        val_threshold = 0.50
        val_f1 = float(f1_score(val_labels_arr,
                                (best_val_probs >= 0.50).astype(int),
                                pos_label=1, zero_division=0))
    print(f'  Val threshold={val_threshold}  Val F1={val_f1:.4f}')

    # ── Test evaluation ──────────────────────────────────────────────────────
    model.load_state_dict(best_state)
    model.to(DEVICE)
    test_probs  = get_probs(model, test_loader)
    te_labels_a = np.array(te_labels)
    test_preds  = (test_probs >= val_threshold).astype(int)
    test_f1     = float(f1_score(te_labels_a, test_preds, pos_label=1, zero_division=0))
    test_prec   = float(precision_score(te_labels_a, test_preds, pos_label=1, zero_division=0))
    test_rec    = float(recall_score(te_labels_a, test_preds, pos_label=1, zero_division=0))

    print(f'  Test  F1={test_f1:.4f}  Precision={test_prec:.4f}  Recall={test_rec:.4f}')

    del model
    torch.cuda.empty_cache()

    return {
        'variant'        : variant_name,
        'val_f1'         : round(val_f1, 4),
        'val_threshold'  : val_threshold,
        'test_f1'        : round(test_f1, 4),
        'test_precision' : round(test_prec, 4),
        'test_recall'    : round(test_rec, 4),
        'best_epoch'     : best_epoch,
    }

## 8. Run All Ablation Variants

Each cell runs one variant. Run them sequentially — estimated ~8 min per variant on a Tesla T4.
Results accumulate in `results`.

In [ ]:
results = []

In [ ]:
# ── Variant 0: Full model (all components ON) ─────────────────────────────────
r = train_and_evaluate(
    'Full model',
    use_metadata=True, use_wrs=True, use_llrd=True,
    use_warmup=True, use_thresh_tune=True,
)
results.append(r)

In [ ]:
# ── Variant 1: Remove metadata prepending ────────────────────────────────────
r = train_and_evaluate(
    'w/o Metadata',
    use_metadata=False, use_wrs=True, use_llrd=True,
    use_warmup=True, use_thresh_tune=True,
)
results.append(r)

In [ ]:
# ── Variant 2: Remove Weighted Random Sampler ─────────────────────────────────
r = train_and_evaluate(
    'w/o WRS',
    use_metadata=True, use_wrs=False, use_llrd=True,
    use_warmup=True, use_thresh_tune=True,
)
results.append(r)

In [ ]:
# ── Variant 3: Remove Grouped LLRD (uniform LR = 2e-5) ────────────────────────
r = train_and_evaluate(
    'w/o LLRD',
    use_metadata=True, use_wrs=True, use_llrd=False,
    use_warmup=True, use_thresh_tune=True,
)
results.append(r)

In [ ]:
# ── Variant 4: Remove warmup (cosine from step 0) ────────────────────────────
r = train_and_evaluate(
    'w/o Warmup',
    use_metadata=True, use_wrs=True, use_llrd=True,
    use_warmup=False, use_thresh_tune=True,
)
results.append(r)

In [ ]:
# ── Variant 5: No threshold tuning (fixed t=0.50) ─────────────────────────────
r = train_and_evaluate(
    'Fixed t=0.50',
    use_metadata=True, use_wrs=True, use_llrd=True,
    use_warmup=True, use_thresh_tune=False,
)
results.append(r)

## 9. Results Table and Figure

In [ ]:
df_res = pd.DataFrame(results)

# Compute delta vs full model
full_test_f1 = df_res.loc[df_res.variant == 'Full model', 'test_f1'].values[0]
df_res['delta_f1'] = (df_res['test_f1'] - full_test_f1).round(4)

# Add known results from hyperparam search
prior = pd.DataFrame([
    {'variant': 'RoBERTa drop=0.0 (best, from log)', 'val_f1': 0.5979, 'val_threshold': 0.87,
     'test_f1': 0.6034, 'test_precision': None, 'test_recall': None, 'best_epoch': None,
     'delta_f1': round(0.6034 - full_test_f1, 4)},
    {'variant': 'RoBERTa drop=0.5 (from log)',       'val_f1': 0.2768, 'val_threshold': None,
     'test_f1': None,   'test_precision': None, 'test_recall': None, 'best_epoch': None,
     'delta_f1': None},
    {'variant': 'Ensemble RoBERTa+DistilBERT (submitted)', 'val_f1': 0.5882, 'val_threshold': 0.50,
     'test_f1': 0.6069, 'test_precision': 0.560, 'test_recall': 0.653, 'best_epoch': None,
     'delta_f1': round(0.6069 - full_test_f1, 4)},
])
df_all = pd.concat([df_res, prior], ignore_index=True)

# Save to CSV
df_all.to_csv(os.path.join(OUT_DIR, 'ablation_results.csv'), index=False)

# Display
display_cols = ['variant', 'val_f1', 'val_threshold', 'test_f1',
                'test_precision', 'test_recall', 'delta_f1', 'best_epoch']
print(df_all[display_cols].to_string(index=False))

In [ ]:
# ── Bar chart: test F1 by variant ───────────────────────────────────────────
plot_df = df_all[df_all['test_f1'].notna()].copy()

# Colour: green for full model / ensemble, red for ablated variants
palette = [
    '#2ecc71' if 'Full' in v or 'Ensemble' in v
    else '#e74c3c'
    for v in plot_df['variant']
]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(plot_df['variant'], plot_df['test_f1'], color=palette, edgecolor='white')

# Annotate bars with F1 value
for bar, val in zip(bars, plot_df['test_f1']):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=9)

# Baseline reference line
ax.axvline(x=0.48, color='black', linestyle='--', linewidth=1.2, label='Official baseline (0.48)')
ax.axvline(x=full_test_f1, color='steelblue', linestyle=':', linewidth=1.2,
           label=f'Full DistilBERT ablation baseline ({full_test_f1:.4f})')

ax.set_xlabel('Test F1 (PCL class)')
ax.set_title('Ablation Study — Component-level Impact on Dev Set F1', fontsize=13)
ax.set_xlim(0, 0.70)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'ablation_bar.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {os.path.join(OUT_DIR, "ablation_bar.png")}')

In [ ]:
# ── Delta chart: F1 drop relative to Full model ──────────────────────────────
delta_df = df_all[
    df_all['test_f1'].notna() &
    df_all['delta_f1'].notna() &
    (df_all['variant'] != 'Full model')
].copy()

palette_delta = ['#2ecc71' if d >= 0 else '#e74c3c' for d in delta_df['delta_f1']]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(delta_df['variant'], delta_df['delta_f1'], color=palette_delta, edgecolor='white')
for bar, val in zip(bars, delta_df['delta_f1']):
    x = bar.get_width() + (0.002 if val >= 0 else -0.002)
    ha = 'left' if val >= 0 else 'right'
    ax.text(x, bar.get_y() + bar.get_height() / 2, f'{val:+.4f}', va='center',
            ha=ha, fontsize=9)

ax.axvline(x=0, color='black', linewidth=1.0)
ax.set_xlabel('ΔF1 vs Full model (positive = improvement over full)')
ax.set_title('F1 Impact of Removing Each Component (Full model baseline)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'ablation_delta.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {os.path.join(OUT_DIR, "ablation_delta.png")}')

## 10. Summary and Key Findings

The cell below prints a plain-English summary of the results to copy into the report.

In [ ]:
print('=' * 60)
print('ABLATION SUMMARY')
print('=' * 60)
print(f'Full DistilBERT baseline test F1 : {full_test_f1:.4f}')
print()

ablated = df_all[df_all['delta_f1'].notna() & (df_all['variant'] != 'Full model')]
ablated_sorted = ablated.sort_values('delta_f1')

for _, row in ablated_sorted.iterrows():
    tf = f"{row['test_f1']:.4f}" if pd.notna(row['test_f1']) else 'N/A'
    d  = f"{row['delta_f1']:+.4f}"
    print(f"  {row['variant']:<45}  test F1={tf}  delta={d}")

print()
print('Most harmful removal (largest F1 drop) ->')
worst = ablated_sorted[ablated_sorted['test_f1'].notna()].iloc[0]
print(f"  {worst['variant']}  (delta={worst['delta_f1']:+.4f})")
print()
print('Results saved to:', os.path.join(OUT_DIR, 'ablation_results.csv'))